# RFM 会员价值分析项目

> 一个基于 Python、Pandas 与 Pyecharts 的会员消费行为分析项目，可用于数据分析课程实践与个人 GitHub 项目展示。

## 项目简介

本项目使用 RFM 模型对会员消费行为进行分析，并比较不同年份会员分群的分布情况。

RFM 由三个指标组成：

- **R — Recency（最近消费时间）**：会员距离分析基准日最近一次消费过去了多少天。间隔越短，通常代表近期活跃度越高。
- **F — Frequency（消费频率）**：会员在分析周期内的订单数量。订单越多，通常代表消费频率越高。
- **M — Monetary（消费金额）**：会员在分析周期内累计消费金额。金额越高，通常代表消费贡献越大。

项目流程：数据读取 → 数据质量检查 → 数据清洗 → 合并年度订单 → 按会员和年份计算 RFM → 分箱评分 → 生成会员分群 → 导出结果 → 可视化与业务解读。

## 项目目标

1. 使用 Pandas 读取 Excel 多工作表数据并完成清洗。
2. 按年度和会员计算 R、F、M 三项指标。
3. 使用分箱方法将 RFM 指标转换为 1—3 分，并拼接为分群编码。
4. 通过图表观察不同年份的会员分群规模和变化。
5. 导出可复用的分析结果，为会员运营提供参考。

## 数据说明

本 Notebook 默认读取 `data/sales.xlsx`。Excel 文件应包含以下工作表：

- `2015`
- `2016`
- `2017`
- `2018`
- `会员等级`

订单工作表至少需要包含字段：`提交日期`、`订单金额`、`订单号`、`会员ID`。如果你的原始文件字段名称不同，请在“数据检查”章节统一修改。

**重要：** 原始销售数据可能包含个人或商业敏感信息。未经授权，不要将原始数据上传到公开 GitHub 仓库。公开仓库建议只提交脱敏后的示例数据、代码和分析说明。

## 目录

1. 环境准备与导入库
2. 加载数据
3. 数据质量检查与预处理
4. 合并年度订单并构造时间差
5. 计算 RFM 指标
6. RFM 分箱与会员分群
7. 导出分析结果
8. 可视化分析
9. 业务结论与改进方向

---

## 1. 环境准备与导入库

在终端或 Notebook 中安装依赖。若你使用 Jupyter，推荐在单元格运行 `%pip install ...`，确保安装到当前 Notebook 内核使用的 Python 环境。

In [ ]:
%pip install pandas numpy openpyxl matplotlib pyecharts

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyecharts.charts import Bar3D
import pyecharts.options as opts
from pyecharts.globals import ThemeType

# 统一图表的字体设置；不同系统的中文字体名称可能不同。
plt.rcParams["axes.unicode_minus"] = False

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_FILE = DATA_DIR / "sales.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("当前项目目录：", PROJECT_DIR.resolve())
print("数据文件路径：", DATA_FILE.resolve())
print("数据文件是否存在：", DATA_FILE.exists())

### 1.1 目录结构建议

把原始 Excel 文件放在项目目录的 `data/` 文件夹下。分析输出统一保存到 `outputs/` 文件夹，便于管理，也避免在代码中写入只适用于某一台电脑的绝对路径。

```text
RFM_GitHub_Project/
├── RFM_会员价值分析.ipynb
├── README.md
├── requirements.txt
├── .gitignore
├── data/
│   └── sales.xlsx       # 原始数据文件
└── outputs/             # 分析生成文件
```

## 2. 加载数据

通过 `pd.read_excel(..., sheet_name=...)` 一次性读取多个工作表，返回以工作表名称为键、DataFrame 为值的字典。

如果数据文件缺失或工作表名称不匹配，先检查文件路径和 Excel 工作表名称。

In [ ]:
sheet_names = ["2015", "2016", "2017", "2018", "会员等级"]

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"没有找到数据文件：{DATA_FILE}\n"
        "请将原始 Excel 文件放到项目目录的 data/sales.xlsx。"
    )

sheet_dict = pd.read_excel(DATA_FILE, sheet_name=sheet_names)

print("已读取工作表：", list(sheet_dict.keys()))
for name, frame in sheet_dict.items():
    print(f"{name}: {frame.shape[0]:,} 行，{frame.shape[1]} 列")

In [ ]:
# 预览 2015 年订单数据
display(sheet_dict["2015"].head())

# 查看列名、数据类型和缺失值概况
sheet_dict["2015"].info()
display(sheet_dict["2015"].isna().sum().sort_values(ascending=False).to_frame("缺失值数量"))

### 2.1 理解数据结构

- `head()`：查看前几行，确认字段内容是否符合预期。
- `info()`：检查每列的数据类型和非空值数量。
- `isna().sum()`：统计各字段缺失数量。
- `describe()`：查看数值列的统计分布，帮助发现异常值。

不要只依赖代码“能运行”来判断数据正确；还要确认日期、订单金额、订单号和会员 ID 的含义及数据类型。

In [ ]:
for name, frame in sheet_dict.items():
    print(f"\n===== {name} =====")
    print("形状：", frame.shape)
    print("列名：", frame.columns.tolist())
    display(frame.head(3))

## 3. 数据质量检查与预处理

本项目沿用原始分析思路：

1. 仅处理 2015—2018 年订单表，会员等级表不参与订单合并。
2. 将订单日期转换为 Pandas 日期类型。
3. 删除缺失记录。
4. 仅保留订单金额大于 1 的订单。
5. 将每年订单表中最大的提交日期作为该年度的分析基准日。

**口径提醒：** 这里的分析基准日是“该年度数据中最大的提交日期”，不一定恰好是 12 月 31 日。如果业务要求严格以自然年年末为基准，应另行设置每年的固定日期。

In [ ]:
required_columns = {"提交日期", "订单金额", "订单号", "会员ID"}
order_sheet_names = ["2015", "2016", "2017", "2018"]

cleaned_sheets = {}

for year_name in order_sheet_names:
    df = sheet_dict[year_name].copy()

    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(
            f"{year_name} 工作表缺少必要字段：{sorted(missing_columns)}"
        )

    # 统一日期和金额的数据类型
    df["提交日期"] = pd.to_datetime(df["提交日期"], errors="coerce")
    df["订单金额"] = pd.to_numeric(df["订单金额"], errors="coerce")

    before_rows = len(df)

    # 只删除 RFM 计算所必需字段存在缺失的记录，
    # 避免因其他非必要字段缺失而无谓丢弃订单。
    df = df.dropna(subset=list(required_columns))
    df = df[df["订单金额"] > 1].copy()

    if df.empty:
        raise ValueError(f"{year_name} 清洗后没有可用订单，请检查原始数据和筛选条件。")

    # 每年最后一笔订单的提交日期作为该年分析基准日
    df["max_year_date"] = df["提交日期"].max()
    df["year"] = int(year_name)

    cleaned_sheets[year_name] = df

    print(
        f"{year_name}：原始 {before_rows:,} 行，"
        f"清洗后 {len(df):,} 行，"
        f"基准日 {df['max_year_date'].iloc[0].date()}"
    )

### 3.1 为什么要清洗数据？

- 缺失的会员 ID 无法归属到具体会员。
- 缺失的订单号可能影响订单数量统计。
- 缺失的日期无法计算最近消费时间。
- 缺失或无法转换的金额不能用于消费金额累计。
- 本项目过滤掉金额小于或等于 1 的订单，属于项目既定业务规则；实际业务中应确认这是否会误删真实订单。

为了让清洗过程可解释，代码输出清洗前后的行数和分析基准日。

## 4. 合并年度订单并构造时间差

将四个年度订单表纵向合并，再构造两个字段：

- `year`：订单所属年份。
- `date_interval`：订单日期距离该年度分析基准日的天数。

例如，某订单距离分析基准日 20 天，则 `date_interval = 20`。对于同一会员同一年内的多笔订单，后续取最小值作为 R 值。

In [ ]:
df_merge = pd.concat(
    [cleaned_sheets[year] for year in order_sheet_names],
    ignore_index=True
)

df_merge["date_interval"] = (
    df_merge["max_year_date"] - df_merge["提交日期"]
).dt.days

print("合并后数据形状：", df_merge.shape)
display(df_merge.head())
display(df_merge[["year", "date_interval", "订单金额"]].describe().T)

## 5. 计算 RFM 指标

按 `year` 和 `会员ID` 分组，计算：

- **R（Recency）**：`date_interval` 的最小值。越小表示越近期有消费。
- **F（Frequency）**：订单号的非空计数，作为订单频率的代理指标。
- **M（Monetary）**：订单金额求和，表示该会员当年累计消费金额。

这里的会员统计单位是“会员—年份”。同一个会员如果在多个年份消费，会在每个年份分别得到一条 RFM 记录。

In [ ]:
rfm_gb = (
    df_merge
    .groupby(["year", "会员ID"], as_index=False)
    .agg(
        r=("date_interval", "min"),
        f=("订单号", "count"),
        m=("订单金额", "sum")
    )
)

print("RFM 记录数：", len(rfm_gb))
display(rfm_gb.head())
display(rfm_gb[["r", "f", "m"]].describe().T)

### 5.1 解释统计结果

查看 `describe()` 输出时重点关注：

- `min` / `max`：最小值和最大值，可用于检查异常值。
- `mean`：均值，可能受少数大额会员影响。
- `50%`：中位数，可与均值比较判断分布是否偏斜。
- `std`：标准差，反映数据离散程度。

R、F、M 的量纲不同，不能直接将原始值相加比较，因此下一步将每个指标分成三个等级。

## 6. RFM 分箱与会员分群

本项目采用三档评分法：

- R：最近消费越近越好，因此天数越小，评分越高。
- F：订单数量越多，评分越高。
- M：累计消费金额越高，评分越高。

项目原始区间如下。区间端点是可调整的业务参数，必须根据实际数据分布核对；若数据超出区间，代码会提醒并将这些记录标记出来，避免悄悄把缺失评分当作有效分群。

In [ ]:
# 项目使用的手动分箱边界；如数据范围变化，应根据 describe() 结果调整。
r_bins = [-1, 79, 255, 365]
f_bins = [0, 2, 5, 130]
m_bins = [1, 69, 1199, 206252]

rfm_gb["r_label"] = pd.cut(
    rfm_gb["r"], bins=r_bins, labels=[3, 2, 1], include_lowest=True
)
rfm_gb["f_label"] = pd.cut(
    rfm_gb["f"], bins=f_bins, labels=[1, 2, 3], include_lowest=True
)
rfm_gb["m_label"] = pd.cut(
    rfm_gb["m"], bins=m_bins, labels=[1, 2, 3], include_lowest=True
)

label_columns = ["r_label", "f_label", "m_label"]
unscored = rfm_gb[rfm_gb[label_columns].isna().any(axis=1)]

print(f"存在未评分记录：{len(unscored):,} 条")
if not unscored.empty:
    display(unscored[["year", "会员ID", "r", "f", "m"]].head(10))
    warnings.warn(
        "部分记录超出分箱范围或无法评分。请根据上面的记录检查并调整 r_bins、f_bins、m_bins。"
    )

In [ ]:
# 只有三项评分都有效的记录才能组成完整的三位 RFM 分群编码
rfm_scored = rfm_gb.dropna(subset=label_columns).copy()

for col in label_columns:
    rfm_scored[col] = rfm_scored[col].astype(int).astype(str)

rfm_scored["rfm_group"] = (
    rfm_scored["r_label"]
    + rfm_scored["f_label"]
    + rfm_scored["m_label"]
)

print("成功评分记录数：", len(rfm_scored))
print("分群编码示例：", sorted(rfm_scored["rfm_group"].unique())[:10])
display(rfm_scored.head())

### 6.1 如何理解分群编码？

例如：

- `333`：R、F、M 评分均高，通常可以视为近期活跃、消费频率较高且金额贡献较高的会员。
- `311`：最近消费评分高，但频率和金额评分低，可能是近期首次或低金额消费的会员。
- `133`：最近消费评分低，但频率和金额评分高，可能是过去贡献较高、近期活跃度下降的会员。
- `111`：三个维度评分都低，可作为低活跃或低贡献群体进一步观察。

这些是基于 RFM 评分的**初步运营解释**，不是对会员行为的确定结论。正式业务分析应结合业务背景、会员生命周期和更多数据验证。

## 7. 导出分析结果

结果默认导出到项目的 `outputs/` 文件夹。这样不依赖某台电脑的绝对路径，也不会在每次运行时覆盖原始数据文件。

本项目将导出 Excel 结果与年度分群统计表。MySQL 导出是可选步骤，默认不执行，避免在公开 Notebook 中暴露数据库密码。

In [ ]:
rfm_output = OUTPUT_DIR / "rfm_member_scores.xlsx"
rfm_scored.to_excel(rfm_output, index=False)

display_data = (
    rfm_scored
    .groupby(["rfm_group", "year"], as_index=False)
    .agg(number=("会员ID", "count"))
)

group_output = OUTPUT_DIR / "rfm_group_year_summary.csv"
display_data.to_csv(group_output, index=False, encoding="utf-8-sig")

print("会员评分结果已保存：", rfm_output.resolve())
print("年度分群汇总已保存：", group_output.resolve())
display(display_data.head())

### 7.1 可选：导出到 MySQL

如果确实需要将结果写入 MySQL，请先在本机创建数据库，并通过环境变量或本地私密配置读取连接信息。**不要将真实密码写进 Notebook 并上传 GitHub。**

下面代码仅作模板，只有配置好环境变量后再运行：

In [ ]:
# 可选步骤：先在终端设置环境变量，再取消下面代码的注释运行。
#
# import os
# from sqlalchemy import create_engine
#
# DB_USER = os.getenv("RFM_DB_USER")
# DB_PASSWORD = os.getenv("RFM_DB_PASSWORD")
# DB_HOST = os.getenv("RFM_DB_HOST", "localhost")
# DB_PORT = os.getenv("RFM_DB_PORT", "3306")
# DB_NAME = os.getenv("RFM_DB_NAME", "rfm_db")
#
# if not DB_USER or not DB_PASSWORD:
#     raise ValueError("请先设置 RFM_DB_USER 和 RFM_DB_PASSWORD 环境变量。")
#
# engine = create_engine(
#     f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4",
#     pool_pre_ping=True,
#     connect_args={"connect_timeout": 10}
# )
#
# rfm_scored.to_sql(
#     "rfm_table",
#     con=engine,
#     index=False,
#     if_exists="replace"
# )
#
# print(pd.read_sql("SELECT COUNT(*) AS total FROM rfm_table", engine))

## 8. 可视化分析

本节生成三类分析结果：

1. 年度 RFM 分群人数的交互式 3D 柱状图。
2. 各年份会员分群数量的堆叠柱状图。
3. 各年份总会员记录数及人数最多的分群汇总。

注意：一个会员可能在多个年份各出现一次，因此这里统计的是“会员—年份记录数”，不一定等于跨年度去重后的会员总人数。

In [ ]:
if rfm_scored.empty:
    raise ValueError("没有可视化的有效评分记录，请先检查分箱区间。")

# 统计各年份、各分群的记录数量
display_data = (
    rfm_scored.groupby(["year", "rfm_group"], as_index=False)
    .agg(number=("会员ID", "count"))
)

years = sorted(display_data["year"].astype(int).unique().tolist())
rfm_groups = sorted(display_data["rfm_group"].astype(str).unique().tolist())

# 补齐不存在的“年份-分群”组合，人数设为 0，方便年度比较
full_index = pd.MultiIndex.from_product(
    [years, rfm_groups], names=["year", "rfm_group"]
)
plot_df = (
    display_data.assign(
        year=display_data["year"].astype(int),
        rfm_group=display_data["rfm_group"].astype(str)
    )
    .set_index(["year", "rfm_group"])["number"]
    .reindex(full_index, fill_value=0)
    .reset_index()
)

display(plot_df.head())

In [ ]:
# 8.1 交互式 3D 柱状图（Pyecharts）
bar3d_data = [
    [years.index(int(row.year)),
     rfm_groups.index(str(row.rfm_group)),
     int(row.number)]
    for row in plot_df.itertuples(index=False)
]

max_count = max(1, int(plot_df["number"].max()))

rfm_chart = (
    Bar3D(
        init_opts=opts.InitOpts(
            theme=ThemeType.WHITE,
            width="1200px",
            height="720px"
        )
    )
    .add(
        series_name="会员记录数",
        data=bar3d_data,
        xaxis3d_opts=opts.Axis3DOpts(
            type_="category",
            data=[str(y) for y in years],
            name="年份"
        ),
        yaxis3d_opts=opts.Axis3DOpts(
            type_="category",
            data=rfm_groups,
            name="RFM 分群"
        ),
        zaxis3d_opts=opts.Axis3DOpts(
            type_="value",
            name="会员记录数"
        ),
        shading="color"
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="年度 RFM 会员分群分布",
            subtitle="柱体高度表示对应年份与分群的会员—年份记录数"
        ),
        visualmap_opts=opts.VisualMapOpts(
            max_=max_count,
            range_color=["#e8f3ff", "#6baed6", "#08519c"],
            is_calculable=True
        ),
        tooltip_opts=opts.TooltipOpts(is_show=True)
    )
)

html_path = OUTPUT_DIR / "rfm_segment_3d.html"
rfm_chart.render(str(html_path))
print("交互式图表已生成：", html_path.resolve())

# 在 Notebook 中显示
rfm_chart.render_notebook()

In [ ]:
# 8.2 堆叠柱状图：比较各年份的分群结构
pivot_df = (
    plot_df.pivot(index="year", columns="rfm_group", values="number")
    .fillna(0)
    .sort_index()
)

ax = pivot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(13, 6),
    width=0.8
)
ax.set_title("各年度 RFM 会员分群记录数")
ax.set_xlabel("年份")
ax.set_ylabel("会员—年份记录数")
ax.legend(title="RFM 分群", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()

stacked_path = OUTPUT_DIR / "rfm_segment_yearly_stacked.png"
plt.savefig(stacked_path, dpi=160, bbox_inches="tight")
plt.show()

print("年度堆叠柱状图已保存：", stacked_path.resolve())

In [ ]:
# 8.3 年度总记录数与各年人数最多的分群
year_total = (
    plot_df.groupby("year", as_index=False)
    .agg(member_year_records=("number", "sum"))
)

top_groups = (
    plot_df.sort_values(["year", "number"], ascending=[True, False])
    .groupby("year", as_index=False)
    .first()[["year", "rfm_group", "number"]]
    .rename(columns={
        "rfm_group": "人数最多的分群",
        "number": "该分群记录数"
    })
)

print("年度会员—年份记录数：")
display(year_total)

print("各年人数最多的分群：")
display(top_groups)

### 8.4 比较首尾年份的分群变化

下面代码比较数据中最早年份和最晚年份的分群记录数。变化率只在首年人数大于 0 时计算；若首年人数为 0，则变化率留空，避免除以零。

In [ ]:
first_year, last_year = min(years), max(years)

compare = plot_df[plot_df["year"].isin([first_year, last_year])].pivot(
    index="rfm_group", columns="year", values="number"
).fillna(0)

# 确保首尾年份列都存在
for y in [first_year, last_year]:
    if y not in compare.columns:
        compare[y] = 0

compare["人数变化"] = compare[last_year] - compare[first_year]
compare["变化率（%）"] = np.where(
    compare[first_year] > 0,
    (compare[last_year] - compare[first_year]) / compare[first_year] * 100,
    np.nan
)
compare = compare.sort_values("人数变化", ascending=False)

print(f"{first_year}—{last_year} 年增长最多的分群：")
display(compare.head(10).round(2))

print(f"{first_year}—{last_year} 年下降最多的分群：")
display(compare.tail(10).sort_values("人数变化").round(2))

## 9. 业务结论与改进方向

运行完上面的分析后，请根据实际输出填写结论，不要在没有查看统计结果前预先声称某类会员一定增长或下降。

### 9.1 建议的结论写作框架

- **会员结构：** 哪些 RFM 分群在各年度占比最高？结构是否发生变化？
- **高价值会员维护：** `333` 等高分群的规模和变化如何？可以考虑会员权益、专属服务和复购激励。
- **活跃度下降预警：** 对于 R 评分低但 F、M 评分较高的群体，可以尝试召回活动，并观察活动前后的回访率。
- **新会员培养：** 对近期有消费但频率或金额较低的群体，可以设计二次购买优惠或相关商品推荐。
- **低贡献群体：** 先通过小规模实验测试触达成本与转化效果，再决定是否扩大投入。

### 9.2 项目局限性

1. RFM 只考虑消费时间、频率和金额，无法独立解释会员的全部行为。
2. 手动分箱阈值依赖当前数据范围；换数据集后应重新评估阈值。
3. 年度分析基准日采用当年数据最大日期，不一定是自然年末。
4. `订单号` 的计数是当前数据口径下的订单数量；若数据中同一订单存在多行商品明细，需要先确认是否应按订单号去重。
5. 本项目按“会员—年份”统计，年度记录数不应直接解释为跨年度去重会员数。
6. 分群变化只能说明数量变化，不能直接证明营销策略导致了变化。

### 9.3 后续可改进方向

- 使用分位数分箱或业务专家确认的阈值，比较不同评分方案的稳定性。
- 增加会员分群占比、年度趋势折线图和 R/F/M 分布图。
- 结合会员等级、商品类别、渠道、地区等字段丰富解释。
- 对召回策略进行 A/B 测试，衡量复购率、客单价和投入产出比。
- 将分析结果制作成独立 Dashboard 或定期更新的数据报告。

---

## 项目完成检查清单

- [ ] 已确认 Excel 工作表与字段名称。
- [ ] 已检查缺失值、日期类型和金额类型。
- [ ] 已确认清洗前后数据量变化合理。
- [ ] 已检查是否存在未评分记录，并根据实际数据调整分箱边界。
- [ ] 已生成会员 RFM 评分表。
- [ ] 已生成交互式 3D 图和年度堆叠柱状图。
- [ ] 已根据真实输出撰写业务结论。
- [ ] 上传 GitHub 前已移除密码、私密数据和本机绝对路径。